In [ ]:
import sys
sys.path.append('../')
import torch
import glob
import os
from PIL import Image
import numpy as np
from optimize_filter.network import AttU_Net
from torchvision import transforms
from optimize_filter.tiny_network import U_Net_tiny

def backdoor_img(input_dir, state_dict_path, output_dir, mean=None, std=None):

    image_files = glob.glob(os.path.join(input_dir, '*.[jJ][pP][eE][gG]'))

    os.makedirs(output_dir, exist_ok=True)
    state_dict = torch.load(state_dict_path, map_location=torch.device('cpu'))
    net = U_Net_tiny(img_ch=3,output_ch=3)
    net.load_state_dict(state_dict['model_state_dict'])
    net = net.eval()
    if 'imagenet' in input_dir:
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])
    else:
        mean = torch.tensor(mean).view(3, 1, 1)
        std = torch.tensor(std).view(3, 1, 1)
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
    for i, image_file in enumerate(image_files):
        if i > 20:
            break
        img = Image.open(image_file)

        tensor_image = transform(img)
        img_backdoor = net(tensor_image.unsqueeze(0))

        if 'imagenet' not in input_dir:

            img_backdoor = img_backdoor * std + mean  # denormalize


        img_backdoor = torch.clamp(img_backdoor, min=0, max=1)

        scaled_image = (img_backdoor.squeeze().detach().numpy() * 255).astype(np.uint8)
        img_backdoor = Image.fromarray(np.transpose(scaled_image, (1, 2, 0)))

        img_backdoor = img_backdoor.convert('RGB')

        filename = os.path.basename(image_file)

        img_backdoor.save(os.path.join(output_dir, filename))

In [ ]:
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)# if the backdoor is optimized on cifar10
std = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1) # if the backdoor is optimized on cifar10
state_dict_path = 'output/cifar10/gtsrb_backdoored_encoder/2025-06-26-13:07:41/unet_filter_200_trained.pt'
input_dir = 'data/gtsrb/test'
output_dir = 'data/gtsrb/2025-06-26-13:07:41'

backdoor_img(input_dir, state_dict_path, output_dir, mean, std)